<a href="https://colab.research.google.com/github/MuwafagQ/Playbook-program/blob/claude%2Fsetup-gpu-video-testing-JhgUH/colab_gpu_test%20(3.2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playbook Soccer Analytics — GPU Test (Google Colab)

**Before running:** Go to `Runtime → Change runtime type` and select **T4 GPU**.

You will need:
- A **Roboflow API key** (free at roboflow.com) stored in Colab Secrets as `ROBOFLOW_API_KEY`
- A short soccer video clip (MP4, ideally 10–30 seconds for a quick test)

**Pipeline highlights (good-baseline-may9):**
- BoTSort tracker + appearance ReID (`yolo11n-cls.pt`)
- IDStabilizer — re-links IDs after occlusions using position + torso appearance
- Color-based team classification (fast, no GPU needed for this step)
- BallSmoother — interpolates missing ball detections
- HomographyStateMachine — holds last good homography through short failures
- KPI summary JSON/CSV alongside the annotated video

In [2]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print('GPU detected:', gpu.stdout.strip())
else:
    print('⚠️  No GPU found.\n'
          'Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')

GPU detected: Tesla T4, 15360 MiB


In [3]:
# ── Cell 2: System packages ───────────────────────────────────────────────────
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev

In [4]:
# ── Cell 3: Python dependencies ───────────────────────────────────────────────
# Swap out Colab's opencv for headless (avoids display-backend conflicts)
!pip uninstall -qqy opencv-python opencv-python-headless 2>/dev/null

!pip install -q \
    'numpy>=2.0.0,<2.4.0' \
    opencv-python-headless==4.10.0.84 \
    onnxruntime==1.20.1 \
    tqdm \
    'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' \
    pydantic-settings==2.4.0 \
    python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' \
    'inference==1.2.2' \
    'ultralytics>=8.4.37,<8.5.0' \
    'lap>=0.5.13,<0.6'

!pip install -q 'transformers>=5.2.0,<5.3.0'

# Roboflow sports library (color-team helper, pitch config, annotators)
!pip install -q git+https://github.com/roboflow/sports.git@main

print('\n✅ All packages installed.')

  Preparing metadata (setup.py) ... done

✅ All packages installed.


In [5]:
!pip install -q pycuda

In [ ]:
# ── Repair Cell: Ensure CUDA-enabled libraries ───────────────────────────────
# If Cell 5 or 8 shows CUDA is unavailable, run this cell and then RESTART RUNTIME.
!pip install --force-reinstall -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu==1.20.1

import torch
print(f"CUDA available after reinstall: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Repair Cell: Ensure CUDA-enabled libraries ───────────────────────────────
# If Cell 5 or 8 shows CUDA is unavailable, run this cell and then RESTART RUNTIME.
!pip install --force-reinstall -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu==1.20.1

import torch
print(f"CUDA available after reinstall: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 27.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 41.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 66.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 415.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 906.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 27.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 839.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 622.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [6]:
# ── Cell 4: Clone the repo ────────────────────────────────────────────────────
# Public repo — no token needed. If private, replace with:
#   !git clone https://<YOUR_TOKEN>@github.com/muwafagq/playbook-program.git /content/playbook
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
!git clone --branch {BRANCH} https://github.com/muwafagq/playbook-program.git /content/playbook
!git pull origin claude/setup-gpu-video-testing-JhgUH
import os, sys
os.chdir('/content/playbook')
sys.path.insert(0, '/content/playbook')
active_branch = !git rev-parse --abbrev-ref HEAD
print('Working dir:', os.getcwd())
print('Branch:', active_branch[0])

fatal: destination path '/content/playbook' already exists and is not an empty directory.
fatal: not a git repository (or any of the parent directories): .git
Working dir: /content/playbook
Branch: claude/setup-gpu-video-testing-JhgUH


In [12]:
import shutil, os, sys, torch
shutil.copy('baseline.env', '.env')

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = 'CCQD6iTsF1B9BnkJHwU6'

# Patch the API key into .env
with open('.env', 'r') as f:
    env_text = f.read()
env_text = env_text.replace('ROBOFLOW_API_KEY=', f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}')
with open('.env', 'w') as f:
    f.write(env_text)

# Export environment variables
from dotenv import dotenv_values
env_vals = dotenv_values('.env')
for k, v in env_vals.items():
    if v is not None: os.environ[k] = v

# FORCE GPU CONFIGURATION
os.environ['DEVICE'] = 'cuda'
os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY

# Attempt to force Roboflow Inference to use CUDA provider
try:
    import inference.core.devices.utils as dev_utils
    import onnxruntime as ort
    dev_utils.GLOBAL_DEVICE = 'cuda'
    # Check if CUDA is actually available to ONNX
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        print("⚠️ ONNX doesn't see CUDA. Speed might be limited.")
except:
    pass

print('\n── Active model config ─────────────────────────────────')
print(f"CUDA Available (PyTorch): {torch.cuda.is_available()}")
print('DEVICE          :', os.environ.get('DEVICE'))
print('────────────────────────────────────────────────────────')


⚠️ ONNX doesn't see CUDA. Speed might be limited.

── Active model config ─────────────────────────────────
CUDA Available (PyTorch): False
DEVICE          : cuda
────────────────────────────────────────────────────────


In [8]:
import torch, gc, sys

# 1. Clear GPU memory and cache
gc.collect()
torch.cuda.empty_cache()

# 2. Force reload the main module and its vision components to pick up CPU changes
import importlib
modules_to_reload = ['main', 'vision.detect', 'vision.utils']

for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

print('✅ Modules reloaded. Ready to run on CPU.')

✅ Modules reloaded. Ready to run on CPU.


In [9]:
# @title
# ── Cell 6: Provide a test video ──────────────────────────────────────────────
# Choose ONE option and comment out the others.

# --- Option A: Upload a local file -------------------------------------------
from google.colab import files as colab_files
print('Select your MP4 file in the dialog below...')
uploaded = colab_files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]
print('Video ready at:', VIDEO_PATH)

# --- Option B: Download a YouTube clip (yt-dlp) ------------------------------
# !pip install -q yt-dlp
# YT_URL = 'https://www.youtube.com/watch?v=REPLACE_ME'
# !yt-dlp -o /content/test_clip.%(ext)s --recode-video mp4 -q "$YT_URL"
# VIDEO_PATH = '/content/test_clip.mp4'

# --- Option C: Mount Google Drive --------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_PATH = '/content/drive/MyDrive/YOUR_FOLDER/your_clip.mp4'

Select your MP4 file in the dialog below...


KeyboardInterrupt: 

In [ ]:
# ── Cell 7 (optional): Trim to first N seconds ────────────────────────────────
# Skip if your clip is already short (< 30 s).
#TRIM_SECONDS = 20
#TRIMMED_PATH = '/content/playbook/test_trimmed.mp4'
#!ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c copy "{TRIMMED_PATH}" -loglevel warning
#VIDEO_PATH = TRIMMED_PATH
#print(f'Trimmed to {TRIM_SECONDS}s → {VIDEO_PATH}')

In [13]:
# ── Cell 8: Run the pipeline (GPU Verification) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, torch, gc
import main
import importlib

# Ensure environment is strictly set to CUDA
os.environ['DEVICE'] = 'cuda'

# Hardware check before running
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected by PyTorch. Please check Runtime type.")

gc.collect()
torch.cuda.empty_cache()

importlib.reload(main)

OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
VIDEO_INPUT = '/content/HILAL-HAZM_match_B_up7.mp4'

print(f"🚀 Starting GPU Pipeline: {VIDEO_INPUT}")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

try:
    main.main(
        source_video=VIDEO_INPUT,
        out_dir=OUT_DIR,
        enable_team=True,
    )
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")

RuntimeError: GPU not detected by PyTorch. Please check Runtime type.

In [ ]:
# ── Cell 9: Preview annotated video ───────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import glob, os

# run.sh writes to outputs_test/<run_id>/; main() used OUT_DIR directly
video_file = OUT_DIR + '/annotated.mp4'
video_bytes = open(video_file, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(video_bytes).decode()
HTML(f'<video width="800" controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
# ── Cell 10: KPI summary ──────────────────────────────────────────────────────
import json, pandas as pd

kpi_json = OUT_DIR + '/kpi_summary.json'
if os.path.exists(kpi_json):
    with open(kpi_json) as f:
        kpi = json.load(f)
    print(json.dumps(kpi, indent=2))
else:
    print('kpi_summary.json not found — check OUT_DIR path')

csv_path = OUT_DIR + '/per_frame_tracks.csv'
df = pd.read_csv(csv_path)
print(f'\nTracking CSV: {len(df):,} rows | {df.frame.nunique()} frames | {df.track_id.nunique()} unique IDs')
df.head(5)

In [ ]:
# ── Cell 11: Download all outputs ─────────────────────────────────────────────
from google.colab import files as colab_files
for fname in ['annotated.mp4', 'per_frame_tracks.csv', 'kpi_summary.json', 'kpi_summary.csv']:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        colab_files.download(fpath)
    else:
        print(f'Skipped (not found): {fpath}')